# Homework 6: Weekly Weather Forecasting with Holt-Winters Exponential Smoothing

Python (statsmodels) reimplementation of the original R analysis. Forecasts a weekly weather
time series 26 weeks into the future using Holt-Winters exponential smoothing, comparing four
combinations of the trend and seasonal components.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

plt.rcParams["figure.figsize"] = (10, 4)


## 1. Load data and select the weather series

In [ ]:
# Load weekly data (place data_week.csv in the same folder as this notebook)
data_week = pd.read_csv("data_week.csv")
print(data_week.columns.tolist())

# Select avg_temp variable
avg_temp = data_week["avg_temp"]

# Build a time-indexed series with weekly frequency (52 periods/year)
avg_temp_ts = pd.Series(
    avg_temp.values,
    index=pd.date_range(start="2020-01-01", periods=len(avg_temp), freq="W"),
    name="avg_temp",
)
avg_temp_ts.head()


## 2. Four Holt-Winters combinations

Toggle trend (`beta`) and seasonal (`gamma`) components on/off, matching the four models from
the R version:

| Model | Trend (beta) | Seasonal (gamma) |
|---|---|---|
| `ets_1` | ON | ON |
| `ets_2` | ON | OFF |
| `ets_3` | OFF | ON |
| `ets_4` | OFF | OFF |


In [ ]:
SEASONAL_PERIODS = 52

def fit_ets(series, trend, seasonal):
    model = ETSModel(
        series,
        error="add",
        trend="add" if trend else None,
        seasonal="add" if seasonal else None,
        seasonal_periods=SEASONAL_PERIODS if seasonal else None,
    )
    return model.fit(disp=False)

# Beta ON / Gamma ON
ets_1 = fit_ets(avg_temp_ts, trend=True, seasonal=True)

# Beta ON / Gamma OFF
ets_2 = fit_ets(avg_temp_ts, trend=True, seasonal=False)

# Beta OFF / Gamma ON
ets_3 = fit_ets(avg_temp_ts, trend=False, seasonal=True)

# Beta OFF / Gamma OFF (simple exponential smoothing)
ets_4 = fit_ets(avg_temp_ts, trend=False, seasonal=False)

models = {
    "Beta ON Gamma ON": ets_1,
    "Beta ON Gamma OFF": ets_2,
    "Beta OFF Gamma ON": ets_3,
    "Beta OFF Gamma OFF": ets_4,
}


## 3. Plot data series, trend, and seasonal components

In [ ]:
for name, fit in models.items():
    states = fit.states  # level, trend, season components over time
    n_plots = 1 + ("trend" in states.columns) + ("seasonal" in states.columns)
    fig, axes = plt.subplots(n_plots, 1, figsize=(10, 3 * n_plots))
    axes = np.atleast_1d(axes)

    axes[0].plot(avg_temp_ts.index, avg_temp_ts.values)
    axes[0].set_title(f"{name}: Data Series")

    i = 1
    if "trend" in states.columns:
        axes[i].plot(states.index, states["trend"])
        axes[i].set_title(f"{name}: Trend Component")
        i += 1
    if "seasonal" in states.columns:
        axes[i].plot(states.index, states["seasonal"])
        axes[i].set_title(f"{name}: Seasonal Component")

    fig.tight_layout()
    plt.show()


## 4. 26-week out-of-sample forecast with 95% confidence bands

In [ ]:
H = 26
ALPHA = 0.05  # 95% CI

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()

for ax, (name, fit) in zip(axes, models.items()):
    pred = fit.get_prediction(
        start=len(avg_temp_ts),
        end=len(avg_temp_ts) + H - 1,
    )
    summary = pred.summary_frame(alpha=ALPHA)

    ax.plot(avg_temp_ts.index, avg_temp_ts.values, label="Observed")
    ax.plot(summary.index, summary["mean"], color="tab:blue", label="Forecast")
    ax.fill_between(
        summary.index,
        summary["pi_lower"],
        summary["pi_upper"],
        color="tab:blue",
        alpha=0.2,
        label="95% CI",
    )
    ax.set_title(f"{name}: 26-Week Forecast")
    ax.set_xlabel("Week")
    ax.set_ylabel("Average Temperature")
    ax.legend(fontsize=8)

fig.tight_layout()
plt.show()


## Findings

We compared the forecasts from the four models for the next 26 weeks. Models with the trend
component enabled produced wider, less stable confidence bands, suggesting the trend term
overfits noise rather than capturing a persistent linear drift in temperature. Models with the
seasonal component enabled recovered the annual temperature cycle and produced tighter, more
plausible forecasts. This suggests a model with `seasonal="add"` and no trend term is the better
fit for a seasonal series like weekly temperature.
